In [32]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')

for name, df in [('so_h', so_h), ('so_l', so_l), ('products', products)]:
    print(name, df.shape)
    print(list(df.columns))
    print(df.head(3))
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
so_h (20000, 11)
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
       so_id customer_id branch_id  order_date delivery_date order_status  \
0  SO-990591       C0070    AHM001  2022-05-24    2022-05-28    Delivered   
1  SO-431510       C0031    KOL001  2022-05-01    2022-05-08    Delivered   
2  SO-495859       C0013    AHM001  2023-10-01    2023-10-02    Delivered   

  payment_terms  total_order_value  total_gst_amount  grand_total  \
0        Net 30            1345720          374439.6    1720159.6   
1       Advance            2194850          609423.0    2804273.0   
2       Advance            2620800          733824.0    3354624.0   

    sales_channel  
0    Counter Sale  
1  Dealer Network  
2          Online  

so_l (130402

In [33]:
delivered_ids = so_h.loc[so_h['order_status'] == 'Delivered', 'so_id']
sales = so_l[so_l['so_id'].isin(delivered_ids)].copy()

print('Delivered orders:', delivered_ids.nunique())
print('Delivered lines:', len(sales))
print('Units:', sales['quantity'].sum())
print('Revenue before GST (line_total):', sales['line_total'].sum())
print('GST (gst_amount):', sales['gst_amount'].sum())

Delivered orders: 18033
Delivered lines: 117656
Units: 1234939
Revenue before GST (line_total): 22958987430
GST (gst_amount): 6295005732.400001


In [34]:
rate = sales.groupby('gst_rate').agg(lines=('line_total', 'size'), revenue=('line_total', 'sum'), gst=('gst_amount', 'sum')).reset_index()
rate['revenue_share_pct'] = (rate['revenue'] / rate['revenue'].sum() * 100).round(2)

print(rate.to_string())
print('Total lines:', rate['lines'].sum())
print('Total revenue:', rate['revenue'].sum())
print()
print('Products per GST rate in the products table:')
print(products['gst_rate'].value_counts())

   gst_rate  lines      revenue           gst  revenue_share_pct
0        18  51119   1335107480  2.403193e+08               5.82
1        28  66537  21623879950  6.054686e+09              94.18
Total lines: 117656
Total revenue: 22958987430

Products per GST rate in the products table:
gst_rate
28    17
18    13
Name: count, dtype: int64


In [35]:
rev = sales.groupby('product_id').agg(units=('quantity', 'sum'), revenue=('line_total', 'sum')).reset_index()
rev = rev.merge(products[['product_id', 'product_name']], on='product_id', how='left')
rev = rev.sort_values('revenue', ascending=False).reset_index(drop=True)
rev['share_pct'] = (rev['revenue'] / rev['revenue'].sum() * 100).round(2)

print('Products:', len(rev))
print('Total revenue:', rev['revenue'].sum())
print('Total units:', rev['units'].sum())
print('Products without a name:', rev['product_name'].isna().sum())
print(rev.to_string())

Products: 30
Total revenue: 22958987430
Total units: 1234939
Products without a name: 0
   product_id  units     revenue                  product_name  share_pct
0        P015  40815  4856985000           Engine Block EB-600      21.16
1        P014  40574  3387929000  Transmission Assembly TA-800      14.76
2        P021  41864  1934116800          Boom Cylinder BC-400       8.42
3        P019  41543  1653411400           Swing Motor SWM-350       7.20
4        P004  40891  1627461800             Track Chain TC-45       7.09
5        P010  42292  1475990800           Turbocharger TB-900       6.43
6        P011  42093  1161766800     Hydraulic Cylinder HC-200       5.06
7        P001  41304  1011948000         Hydraulic Pump HP-300       4.41
8        P030  41595   910930500              Fuel Pump FP-350       3.97
9        P022  41297   697919300           Control Valve CV-75       3.04
10       P009  40547   604150300          Starter Motor SM-700       2.63
11       P023  40276   5

In [36]:
rev['cum_share'] = rev['revenue'].cumsum() / rev['revenue'].sum() * 100
rev['cum_before'] = rev['cum_share'] - rev['revenue'] / rev['revenue'].sum() * 100

def abc(x):
    if x < 80:
        return 'A'
    elif x < 95:
        return 'B'
    else:
        return 'C'

rev['abc_class'] = rev['cum_before'].apply(abc)
rev['cum_share'] = rev['cum_share'].round(2)
rev['cum_before'] = rev['cum_before'].round(2)

print(rev[['product_id', 'product_name', 'share_pct', 'cum_share', 'abc_class']].to_string())
print()
summary = rev.groupby('abc_class').agg(products=('product_id', 'count'), revenue=('revenue', 'sum')).reset_index()
summary['revenue_share_pct'] = (summary['revenue'] / summary['revenue'].sum() * 100).round(2)
print(summary.to_string())
print('Total revenue:', summary['revenue'].sum())

   product_id                  product_name  share_pct  cum_share abc_class
0        P015           Engine Block EB-600      21.16      21.16         A
1        P014  Transmission Assembly TA-800      14.76      35.91         A
2        P021          Boom Cylinder BC-400       8.42      44.34         A
3        P019           Swing Motor SWM-350       7.20      51.54         A
4        P004             Track Chain TC-45       7.09      58.63         A
5        P010           Turbocharger TB-900       6.43      65.05         A
6        P011     Hydraulic Cylinder HC-200       5.06      70.11         A
7        P001         Hydraulic Pump HP-300       4.41      74.52         A
8        P030              Fuel Pump FP-350       3.97      78.49         A
9        P022           Control Valve CV-75       3.04      81.53         A
10       P009          Starter Motor SM-700       2.63      84.16         B
11       P023             Idler Wheel IW-55       2.27      86.43         B
12       P00

In [37]:
rev = rev.merge(products[['product_id', 'gst_rate', 'unit_price']], on='product_id', how='left')
rev['avg_price_per_unit'] = (rev['revenue'] / rev['units']).round(2)

print('Rows:', len(rev))
print(rev[['product_id', 'product_name', 'abc_class', 'gst_rate', 'unit_price', 'avg_price_per_unit']].to_string())
print()
print(rev.groupby(['abc_class', 'gst_rate']).size())

Rows: 30
   product_id                  product_name abc_class  gst_rate  unit_price  avg_price_per_unit
0        P015           Engine Block EB-600         A        28      119000            119000.0
1        P014  Transmission Assembly TA-800         A        28       83500             83500.0
2        P021          Boom Cylinder BC-400         A        28       46200             46200.0
3        P019           Swing Motor SWM-350         A        28       39800             39800.0
4        P004             Track Chain TC-45         A        28       39800             39800.0
5        P010           Turbocharger TB-900         A        28       34900             34900.0
6        P011     Hydraulic Cylinder HC-200         A        28       27600             27600.0
7        P001         Hydraulic Pump HP-300         A        28       24500             24500.0
8        P030              Fuel Pump FP-350         A        28       21900             21900.0
9        P022           Control

In [38]:
cols_l = ['so_id', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount']
print('Empty values in so_l (columns used):')
print(so_l[cols_l].isnull().sum())
print()
print('Empty values in so_h (so_id, order_status):')
print(so_h[['so_id', 'order_status']].isnull().sum())
print()
print('Empty values in products (columns used):')
print(products[['product_id', 'product_name', 'unit_price', 'gst_rate']].isnull().sum())
print()
for c in ['quantity', 'unit_price', 'line_total']:
    print(c, '| min', so_l[c].min(), '| max', so_l[c].max(), '| zero:', (so_l[c] == 0).sum(), '| negative:', (so_l[c] < 0).sum())

Empty values in so_l (columns used):
so_id         0
product_id    0
quantity      0
unit_price    0
gst_rate      0
line_total    0
gst_amount    0
dtype: int64

Empty values in so_h (so_id, order_status):
so_id           0
order_status    0
dtype: int64

Empty values in products (columns used):
product_id      0
product_name    0
unit_price      0
gst_rate        0
dtype: int64

quantity | min 1 | max 20 | zero: 0 | negative: 0
unit_price | min 310 | max 119000 | zero: 0 | negative: 0
line_total | min 310 | max 2380000 | zero: 0 | negative: 0


In [39]:
cols_l = ['so_id', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount']
print('Delivered lines checked:', len(sales))
print()
print('Empty values in Delivered lines (columns used):')
print(sales[cols_l].isnull().sum())
print()
for c in ['quantity', 'unit_price', 'line_total']:
    print(c, '| min', sales[c].min(), '| max', sales[c].max(), '| zero:', (sales[c] == 0).sum(), '| negative:', (sales[c] < 0).sum())

Delivered lines checked: 117656

Empty values in Delivered lines (columns used):
so_id         0
product_id    0
quantity      0
unit_price    0
gst_rate      0
line_total    0
gst_amount    0
dtype: int64

quantity | min 1 | max 20 | zero: 0 | negative: 0
unit_price | min 310 | max 119000 | zero: 0 | negative: 0
line_total | min 310 | max 2380000 | zero: 0 | negative: 0


In [40]:
print('Table used: sales (Delivered lines), rows =', len(sales))
chk = sales.merge(products[['product_id', 'unit_price']], on='product_id', how='left', suffixes=('', '_list'))
print('Lines checked:', len(chk))
print('Lines with no list price found:', chk['unit_price_list'].isnull().sum())
print('Lines sold at a different price than the list price:', (chk['unit_price'] != chk['unit_price_list']).sum())
print()

dates = so_h[['so_id', 'order_date']]
m = sales.merge(dates, on='so_id', how='left')
print('Rows after adding order dates:', len(m))
m['month'] = pd.to_datetime(m['order_date']).dt.to_period('M')
months_per_product = m.groupby('product_id')['month'].nunique()
print('Months in the data:', m['month'].nunique())
print('Months with sales per product: min', months_per_product.min(), '| max', months_per_product.max())

Table used: sales (Delivered lines), rows = 117656
Lines checked: 117656
Lines with no list price found: 0
Lines sold at a different price than the list price: 0

Rows after adding order dates: 117656
Months in the data: 72
Months with sales per product: min 72 | max 72


In [41]:
print('Table used: sales (Delivered lines), rows =', len(sales))
key = ['so_id', 'product_id']
sales['n_lines'] = sales.groupby(key)['line_number'].transform('size')
rep = sales[sales['n_lines'] >= 2]
print('Order-product combos on 2+ lines:', rep.groupby(key).ngroups)
print('Lines inside those combos:', len(rep))

exact = sales.duplicated(subset=['so_id', 'product_id', 'quantity', 'unit_price', 'line_total'], keep=False)
print('Lines that are exact copies of another line (ignoring line_number):', exact.sum())

p = sales['quantity'].value_counts(normalize=True)
chance = (p ** 2).sum()
print('Chance that two random lines share the same quantity: %.2f%%' % (chance * 100))

two = sales[sales['n_lines'] == 2]
g = two.groupby(key)['quantity'].nunique()
same = (g == 1).sum()
print('Combos with exactly 2 lines:', len(g))
print('Of these, same quantity:', same, '= %.2f%%' % (same / len(g) * 100))

Table used: sales (Delivered lines), rows = 117656
Order-product combos on 2+ lines: 12184
Lines inside those combos: 25422
Lines that are exact copies of another line (ignoring line_number): 1374
Chance that two random lines share the same quantity: 5.00%
Combos with exactly 2 lines: 11202
Of these, same quantity: 547 = 4.88%


In [42]:
print('Table used: sales (Delivered lines), rows =', len(sales))
print('Lines sharing the same so_id and line_number with another line:', sales.duplicated(subset=['so_id', 'line_number'], keep=False).sum())
print('Lines identical in every column to another line:', sales.duplicated(keep=False).sum())
print()
print('Products in rev:', len(rev), '| unique product_id:', rev['product_id'].nunique())
print('Products with the same revenue as another product:', rev['revenue'].duplicated(keep=False).sum())
print('Products table: rows', len(products), '| unique product_id:', products['product_id'].nunique())

Table used: sales (Delivered lines), rows = 117656
Lines sharing the same so_id and line_number with another line: 0
Lines identical in every column to another line: 0

Products in rev: 30 | unique product_id: 30
Products with the same revenue as another product: 0
Products table: rows 30 | unique product_id: 30


In [43]:
print('Table used: sales (Delivered lines), rows =', len(sales))
key = ['so_id', 'product_id']
big = sales[sales['n_lines'] >= 3]
print('Combos with 3 or more lines:', big.groupby(key).ngroups)
print('Lines inside them:', len(big))
print('Largest number of lines in one combo:', big['n_lines'].max())

pairs_total = big.groupby(key).size().apply(lambda n: n * (n - 1) // 2).sum()
same_counts = big.groupby(key + ['quantity']).size()
pairs_same = (same_counts * (same_counts - 1) // 2).sum()
p = sales['quantity'].value_counts(normalize=True)
chance = (p ** 2).sum()
print('Pairs of lines inside these combos:', pairs_total)
print('Pairs with the same quantity:', pairs_same, '= %.2f%%' % (pairs_same / pairs_total * 100))
print('Expected by chance: about', round(pairs_total * chance), '= %.2f%%' % (chance * 100))

Table used: sales (Delivered lines), rows = 117656
Combos with 3 or more lines: 982
Lines inside them: 3018
Largest number of lines in one combo: 6
Pairs of lines inside these combos: 3167
Pairs with the same quantity: 143 = 4.52%
Expected by chance: about 158 = 5.00%


In [44]:
print('Table used: sales (Delivered lines), rows =', len(sales))
print('Delivered header rows:', len(so_h[so_h['order_status'] == 'Delivered']), '| unique Delivered order IDs:', delivered_ids.nunique())
print('Lines:', len(sales), '| Units:', sales['quantity'].sum(), '| Revenue:', sales['line_total'].sum())
print('Revenue per product table total:', rev['revenue'].sum())
print()
print('Lines where line_total is not quantity x unit_price:', (sales['line_total'] != sales['quantity'] * sales['unit_price']).sum())
print()
by_order = sales.groupby('so_id')['line_total'].sum().rename('lines_sum')
hdr = so_h[so_h['order_status'] == 'Delivered'][['so_id', 'total_order_value']].merge(by_order, on='so_id', how='left')
print('Delivered orders compared:', len(hdr))
print('Orders with no lines:', hdr['lines_sum'].isnull().sum())
print('Orders where header total_order_value differs from the sum of its lines:', (hdr['total_order_value'] != hdr['lines_sum']).sum())
print('Header total_order_value (Delivered orders):', hdr['total_order_value'].sum())
print('Sum of the lines (Delivered orders):', hdr['lines_sum'].sum())

Table used: sales (Delivered lines), rows = 117656
Delivered header rows: 18033 | unique Delivered order IDs: 18033
Lines: 117656 | Units: 1234939 | Revenue: 22958987430
Revenue per product table total: 22958987430

Lines where line_total is not quantity x unit_price: 0

Delivered orders compared: 18033
Orders with no lines: 0
Orders where header total_order_value differs from the sum of its lines: 0
Header total_order_value (Delivered orders): 22958987430
Sum of the lines (Delivered orders): 22958987430


In [45]:
print('Table used: sales (Delivered lines), rows =', len(sales))
sales['calc_revenue'] = sales['quantity'] * sales['unit_price']
alt = sales.pivot_table(index='product_id', values='calc_revenue', aggfunc='sum')['calc_revenue']

chk = rev.set_index('product_id')[['revenue', 'share_pct', 'cum_share', 'abc_class']].copy()
chk['alt_revenue'] = alt
print('Products compared:', len(chk))
print('Products where revenue differs between the two methods:', (chk['revenue'] != chk['alt_revenue']).sum())
print('Total revenue, method 2:', alt.sum())

alt_sorted = alt.sort_values(ascending=False)
alt_share = alt_sorted / alt_sorted.sum() * 100
alt_cum = alt_share.cumsum()
alt_before = alt_cum - alt_share
chk['alt_class'] = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in alt_before], index=alt_sorted.index)
chk['alt_share'] = alt_share.round(2)
chk['alt_cum'] = alt_cum.round(2)
print('Products where share differs by more than 0.01:', ((chk['share_pct'] - chk['alt_share']).abs() > 0.01).sum())
print('Products where cumulative share differs by more than 0.01:', ((chk['cum_share'] - chk['alt_cum']).abs() > 0.01).sum())
print('Products where class differs between the two methods:', (chk['abc_class'] != chk['alt_class']).sum())

Table used: sales (Delivered lines), rows = 117656
Products compared: 30
Products where revenue differs between the two methods: 0
Total revenue, method 2: 22958987430
Products where share differs by more than 0.01: 0
Products where cumulative share differs by more than 0.01: 0
Products where class differs between the two methods: 0


In [46]:
print('Table used: rev (30 products from the Delivered lines)')
r = rev[['product_id', 'product_name', 'share_pct', 'cum_share', 'abc_class']].copy()
cum_exact = rev['revenue'].cumsum() / rev['revenue'].sum() * 100
r['class_rule2'] = ['A' if x <= 80 else 'B' if x <= 95 else 'C' for x in cum_exact]
changed = r[r['abc_class'] != r['class_rule2']]
print('Products whose class changes under the other rule:', len(changed))
print(changed[['product_id', 'product_name', 'cum_share', 'abc_class', 'class_rule2']].to_string())
print()
for col in ['abc_class', 'class_rule2']:
    s = rev.assign(c=r[col]).groupby('c').agg(products=('product_id', 'count'), revenue=('revenue', 'sum'))
    s['share_pct'] = (s['revenue'] / s['revenue'].sum() * 100).round(2)
    print('Groups with', col)
    print(s.to_string())
    print()

Table used: rev (30 products from the Delivered lines)
Products whose class changes under the other rule: 2
   product_id          product_name  cum_share abc_class class_rule2
9        P022   Control Valve CV-75      81.53         A           B
16       P005  Fuel Injector FI-220      95.60         B           C

Groups with abc_class
   products      revenue  share_pct
c                                  
A        10  18718459400      81.53
B         7   3229679500      14.07
C        13   1010848530       4.40

Groups with class_rule2
   products      revenue  share_pct
c                                  
A         9  18020540100      78.49
B         7   3585542800      15.62
C        14   1352904530       5.89



In [47]:
print('Table used: sales (Delivered lines), rows =', len(sales))
g = sales.groupby('product_id')['line_total']
ratio = g.max() / g.mean()
print('Largest line / average line per product: lowest %.2f, highest %.2f' % (ratio.min(), ratio.max()))
print('Largest single line anywhere:', sales['line_total'].max(), '= %.3f%% of all revenue' % (sales['line_total'].max() / sales['line_total'].sum() * 100))
print('Lines with the maximum quantity (20):', (sales['quantity'] == 20).sum(), '= %.2f%% of lines' % ((sales['quantity'] == 20).mean() * 100))
print()

trim = sales[sales['quantity'] < 20]
t = trim.groupby('product_id')['line_total'].sum().sort_values(ascending=False)
share = t / t.sum() * 100
before = share.cumsum() - share
t_class = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in before], index=t.index)
cmp = rev.set_index('product_id')[['product_name', 'abc_class']].join(t_class.rename('class_without_big_lines'))
print('Rows left after removing the largest-quantity lines:', len(trim))
print('Products whose class changes:', (cmp['abc_class'] != cmp['class_without_big_lines']).sum())
print(cmp[cmp['abc_class'] != cmp['class_without_big_lines']].to_string())

Table used: sales (Delivered lines), rows = 117656
Largest line / average line per product: lowest 1.86, highest 1.95
Largest single line anywhere: 2380000 = 0.010% of all revenue
Lines with the maximum quantity (20): 5852 = 4.97% of lines

Rows left after removing the largest-quantity lines: 111804
Products whose class changes: 0
Empty DataFrame
Columns: [product_name, abc_class, class_without_big_lines]
Index: []


In [48]:
print('Table used: sales (Delivered lines), rows =', len(sales))
m = sales.merge(so_h[['so_id', 'order_date']], on='so_id', how='left')
print('Rows after adding order dates:', len(m))
print('Lines with no order date:', m['order_date'].isnull().sum())
m['order_date'] = pd.to_datetime(m['order_date'])
print('Order dates from', m['order_date'].min().date(), 'to', m['order_date'].max().date())
m['month'] = m['order_date'].dt.to_period('M')
m['year'] = m['order_date'].dt.year
per_month = m.groupby('month').size()
print('Months:', len(per_month), '| first', per_month.index.min(), '| last', per_month.index.max())
print('Lines per month: lowest', per_month.min(), '| median', int(per_month.median()), '| highest', per_month.max())
print('First month lines:', per_month.iloc[0], '| last month lines:', per_month.iloc[-1])
print()
by_year = m.groupby('year').agg(lines=('line_total', 'size'), units=('quantity', 'sum'), revenue=('line_total', 'sum'))
print(by_year.to_string())
print('Years total lines:', by_year['lines'].sum(), '| revenue:', by_year['revenue'].sum())

Table used: sales (Delivered lines), rows = 117656
Rows after adding order dates: 117656
Lines with no order date: 0
Order dates from 2019-01-01 to 2024-12-31
Months: 72 | first 2019-01 | last 2024-12
Lines per month: lowest 1334 | median 1614 | highest 2126
First month lines: 1679 | last month lines: 1488

      lines   units     revenue
year                           
2019  19710  207925  3870016600
2020  20205  212007  3917532120
2021  19745  208269  3852479620
2022  19264  202242  3816288220
2023  19606  204638  3824675310
2024  19126  199858  3677995560
Years total lines: 117656 | revenue: 22958987430


In [49]:
print('Table used: sales (Delivered lines), rows =', len(sales))
m = sales.merge(so_h[['so_id', 'order_date']], on='so_id', how='left')
m['year'] = pd.to_datetime(m['order_date']).dt.year
m['period'] = m['year'].apply(lambda y: '2019-21' if y <= 2021 else '2022-24')
p = m.groupby('period').agg(units=('quantity', 'sum'), revenue=('line_total', 'sum'))
print(p.to_string())
print('Units change: %.2f%%' % ((p.loc['2022-24', 'units'] / p.loc['2019-21', 'units'] - 1) * 100))
print('Revenue change: %.2f%%' % ((p.loc['2022-24', 'revenue'] / p.loc['2019-21', 'revenue'] - 1) * 100))
print()
pp = m.pivot_table(index='product_id', columns='period', values='line_total', aggfunc='sum')
pp['change'] = pp['2022-24'] - pp['2019-21']
pp = pp.join(rev.set_index('product_id')[['product_name', 'abc_class']])
print('Sum of product changes:', pp['change'].sum(), '| total revenue change:', p.loc['2022-24', 'revenue'] - p.loc['2019-21', 'revenue'])
print()
print('Five biggest falls in revenue:')
print(pp.sort_values('change')[['product_name', 'abc_class', 'change']].head(5).to_string())
print('Five biggest rises in revenue:')
print(pp.sort_values('change')[['product_name', 'abc_class', 'change']].tail(5).to_string())
print()
print('Revenue change by class:')
print(pp.groupby('abc_class')['change'].sum().to_string())

Table used: sales (Delivered lines), rows = 117656
          units      revenue
period                      
2019-21  628201  11640028340
2022-24  606738  11318959090
Units change: -3.42%
Revenue change: -2.76%

Sum of product changes: -321069250 | total revenue change: -321069250

Five biggest falls in revenue:
                    product_name abc_class    change
product_id                                          
P019         Swing Motor SWM-350         A -55919000
P015         Engine Block EB-600         A -52003000
P010         Turbocharger TB-900         A -45439800
P021        Boom Cylinder BC-400         A -44906400
P004           Track Chain TC-45         A -29093800
Five biggest rises in revenue:
                            product_name abc_class   change
product_id                                                 
P029               Pressure Sensor PS-90         C  1082300
P024                  Track Roller TR-60         B  1421400
P030                    Fuel Pump FP-350    

In [50]:
print('Table used: sales (Delivered lines), rows =', len(sales))
m = sales.merge(so_h[['so_id', 'order_date']], on='so_id', how='left')
m['year'] = pd.to_datetime(m['order_date']).dt.year
out = pd.DataFrame({'product_name': rev.set_index('product_id')['product_name'], 'overall': rev.set_index('product_id')['abc_class']})
for y, grp in m.groupby('year'):
    t = grp.groupby('product_id')['line_total'].sum().sort_values(ascending=False)
    share = t / t.sum() * 100
    before = share.cumsum() - share
    out[str(y)] = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in before], index=t.index)
years = [c for c in out.columns if c.isdigit()]
diff = out[years].ne(out['overall'], axis=0)
print('Products per year with a class different from the overall class:')
print(diff.sum().to_string())
print()
print('Number of products in each class per year:')
print(out[years].apply(lambda c: c.value_counts()).fillna(0).astype(int).to_string())
print()
print('Products whose class differs from the overall class in at least one year:', (diff.sum(axis=1) > 0).sum())
print(out[diff.sum(axis=1) > 0].to_string())

Table used: sales (Delivered lines), rows = 117656
Products per year with a class different from the overall class:
2019    0
2020    2
2021    0
2022    0
2023    0
2024    0

Number of products in each class per year:
   2019  2020  2021  2022  2023  2024
C    13    13    13    13    13    13
A    10    10    10    10    10    10
B     7     7     7     7     7     7

Products whose class differs from the overall class in at least one year: 2
                    product_name overall 2019 2020 2021 2022 2023 2024
product_id                                                            
P022         Control Valve CV-75       A    A    B    A    A    A    A
P009        Starter Motor SM-700       B    B    A    B    B    B    B


In [51]:
print('Table used: sales (Delivered lines), rows =', len(sales))
m = sales.merge(so_h[['so_id', 'order_date']], on='so_id', how='left')
m['year'] = pd.to_datetime(m['order_date']).dt.year
two = m[m['product_id'].isin(['P022', 'P009'])]
t = two.pivot_table(index='year', columns='product_id', values=['quantity', 'line_total'], aggfunc='sum')
t.columns = [a + '_' + b for a, b in t.columns]
t['P009_units_per_P022_unit'] = (t['quantity_P009'] / t['quantity_P022']).round(3)
t['P009_minus_P022_revenue'] = t['line_total_P009'] - t['line_total_P022']
print(t.to_string())
print()
pr = products.set_index('product_id')['unit_price']
print('Price of one piece: P022 =', pr['P022'], '| P009 =', pr['P009'])
print('Pieces of P009 needed per piece of P022 to earn the same revenue: %.3f' % (pr['P022'] / pr['P009']))

Table used: sales (Delivered lines), rows = 117656
      line_total_P009  line_total_P022  quantity_P009  quantity_P022  P009_units_per_P022_unit  P009_minus_P022_revenue
year                                                                                                                   
2019        104508600        124637500           7014           7375                     0.951                -20128900
2020        111899000        107129100           7510           6339                     1.185                  4769900
2021         98235700        119094300           6593           7047                     0.936                -20858600
2022         96894700        113568000           6503           6720                     0.968                -16673300
2023         98295300        121443400           6597           7186                     0.918                -23148100
2024         94317000        112047000           6330           6630                     0.955               

In [52]:
print('Table used: so_h (all orders) and so_l (all lines), before any filtering')
allp = so_l.merge(so_h[['so_id', 'order_status']], on='so_id', how='left')
print('Lines after adding status:', len(allp), '| lines with no status:', allp['order_status'].isnull().sum())
by_status = allp.groupby('order_status').agg(lines=('line_total', 'size'), units=('quantity', 'sum'), revenue=('line_total', 'sum'))
by_status['orders'] = so_h.groupby('order_status').size()
by_status['revenue_share_pct'] = (by_status['revenue'] / by_status['revenue'].sum() * 100).round(2)
print(by_status.to_string())
print('Totals: orders', by_status['orders'].sum(), '| lines', by_status['lines'].sum(), '| revenue', by_status['revenue'].sum())
print()
t_all = so_l.groupby('product_id')['line_total'].sum().sort_values(ascending=False)
share = t_all / t_all.sum() * 100
before = share.cumsum() - share
c_all = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in before], index=t_all.index)
cmp = rev.set_index('product_id')[['product_name', 'abc_class']].join(c_all.rename('class_all_statuses'))
print('Products whose class changes if all statuses are included:', (cmp['abc_class'] != cmp['class_all_statuses']).sum())
print(cmp[cmp['abc_class'] != cmp['class_all_statuses']].to_string())

Table used: so_h (all orders) and so_l (all lines), before any filtering
Lines after adding status: 130402 | lines with no status: 0
               lines    units      revenue  orders  revenue_share_pct
order_status                                                         
Cancelled      12746   133595   2488405130    1967               9.78
Delivered     117656  1234939  22958987430   18033              90.22
Totals: orders 20000 | lines 130402 | revenue 25447392560

Products whose class changes if all statuses are included: 0
Empty DataFrame
Columns: [product_name, abc_class, class_all_statuses]
Index: []


In [53]:
print('Table used: sales (Delivered lines), rows =', len(sales))
sb = sales.merge(so_h[['so_id', 'branch_id']], on='so_id', how='left')
print('Rows after adding branch:', len(sb), '| lines with no branch:', sb['branch_id'].isnull().sum())
out = pd.DataFrame({'product_name': rev.set_index('product_id')['product_name'], 'overall': rev.set_index('product_id')['abc_class']})
for br, grp in sb.groupby('branch_id'):
    t = grp.groupby('product_id')['line_total'].sum().sort_values(ascending=False)
    share = t / t.sum() * 100
    before = share.cumsum() - share
    out[br] = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in before], index=t.index)
branches = [c for c in out.columns if c not in ('product_name', 'overall')]
diff = out[branches].ne(out['overall'], axis=0)
print()
print('Revenue per branch:')
print(sb.groupby('branch_id')['line_total'].sum().to_string())
print()
print('Products per branch with a class different from the overall class:')
print(diff.sum().to_string())
print()
print('Number of products in each class per branch:')
print(out[branches].apply(lambda c: c.value_counts()).fillna(0).astype(int).to_string())
print()
print('Products whose class differs from the overall class in at least one branch:', (diff.sum(axis=1) > 0).sum())
print(out[diff.sum(axis=1) > 0].to_string())

Table used: sales (Delivered lines), rows = 117656
Rows after adding branch: 117656 | lines with no branch: 0

Revenue per branch:
branch_id
AHM001    3986314570
CHN001    4682202010
DEL001    3500031300
HYD001    3077048220
KOL001    4339861320
PUN001    3373530010

Products per branch with a class different from the overall class:
AHM001    0
CHN001    0
DEL001    0
HYD001    0
KOL001    2
PUN001    0

Number of products in each class per branch:
   AHM001  CHN001  DEL001  HYD001  KOL001  PUN001
C      13      13      13      13      13      13
A      10      10      10      10      10      10
B       7       7       7       7       7       7

Products whose class differs from the overall class in at least one branch: 2
                    product_name overall AHM001 CHN001 DEL001 HYD001 KOL001 PUN001
product_id                                                                        
P005        Fuel Injector FI-220       B      B      B      B      B      C      B
P025           Cabi

In [54]:
print('Table used: sales (Delivered lines), rows =', len(sales))
sb = sales.merge(so_h[['so_id', 'branch_id']], on='so_id', how='left')
two = sb[sb['product_id'].isin(['P005', 'P025'])]
t = two.pivot_table(index='branch_id', columns='product_id', values=['quantity', 'line_total'], aggfunc='sum')
t.columns = [a + '_' + b for a, b in t.columns]
t['P025_units_per_P005_unit'] = (t['quantity_P025'] / t['quantity_P005']).round(3)
t['P025_minus_P005_revenue'] = t['line_total_P025'] - t['line_total_P005']
print(t.to_string())
print()
pr = products.set_index('product_id')['unit_price']
print('Price of one piece: P005 =', pr['P005'], '| P025 =', pr['P025'])
print('Pieces of P025 needed per piece of P005 to earn the same revenue: %.3f' % (pr['P005'] / pr['P025']))

Table used: sales (Delivered lines), rows = 117656
           line_total_P005  line_total_P025  quantity_P005  quantity_P025  P025_units_per_P005_unit  P025_minus_P005_revenue
branch_id                                                                                                                   
AHM001            60485100         54745900           7158           6974                     0.974                 -5739200
CHN001            70709600         67219550           8368           8563                     1.023                 -3490050
DEL001            52677300         45514300           6234           5798                     0.930                 -7163000
HYD001            47852350         46919450           5663           5977                     1.055                  -932900
KOL001            61270950         62454600           7251           7956                     1.097                  1183650
PUN001            49060700         45710550           5806           5823 

In [55]:
print('Rebuilding from the raw CSV files (fresh read, not the notebook tables)')
raw_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
raw_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
raw_p = pd.read_csv(BASE_PATH + 'products.csv')
print('Raw rows: header', len(raw_h), '| lines', len(raw_l), '| products', len(raw_p))

ok = set(raw_h.loc[raw_h['order_status'] == 'Delivered', 'so_id'])
d = raw_l[raw_l['so_id'].isin(ok)].copy()
d['r'] = d['quantity'] * d['unit_price']
print('Delivered orders:', len(ok), '| Delivered lines:', len(d), '| units:', d['quantity'].sum(), '| revenue:', d['r'].sum())

rr = d.groupby('product_id')['r'].sum().sort_values(ascending=False)
sh = rr / rr.sum() * 100
bef = sh.cumsum() - sh
cl = pd.Series(['A' if x < 80 else 'B' if x < 95 else 'C' for x in bef], index=rr.index)

saved = rev.set_index('product_id')
print()
print('Products rebuilt from raw files:', len(rr))
print('Products where revenue differs from the saved table:', (rr != saved['revenue'].reindex(rr.index)).sum())
print('Products where class differs from the saved table:', (cl != saved['abc_class'].reindex(cl.index)).sum())

print()
key = ['so_id', 'line_number']
cols = ['product_id', 'quantity', 'unit_price', 'line_total']
a = d[key + cols].set_index(key).sort_index()
b = sales[key + cols].set_index(key).sort_index()
print('Lines in raw rebuild:', len(a), '| lines in saved sales table:', len(b))
print('Lines only in the raw rebuild:', len(a.index.difference(b.index)))
print('Lines only in the saved table:', len(b.index.difference(a.index)))
both = a.index.intersection(b.index)
print('Lines compared field by field:', len(both))
print('Lines where product, quantity, price or line total differs:', (a.loc[both] != b.loc[both]).any(axis=1).sum())

print()
for pid in ['P015', 'P022', 'P005', 'P026']:
    x = d[d['product_id'] == pid]
    print(pid, saved.loc[pid, 'product_name'])
    print('  raw files : lines', len(x), '| units', x['quantity'].sum(), '| revenue', x['r'].sum(), '| share %.2f%%' % sh[pid], '| class', cl[pid])
    print('  saved     : units', saved.loc[pid, 'units'], '| revenue', saved.loc[pid, 'revenue'], '| share', saved.loc[pid, 'share_pct'], '| class', saved.loc[pid, 'abc_class'])

print()
oid = 'SO-990591'
print('Order', oid, '| header status:', raw_h.loc[raw_h['so_id'] == oid, 'order_status'].iloc[0], '| header total_order_value:', raw_h.loc[raw_h['so_id'] == oid, 'total_order_value'].iloc[0])
print(raw_l[raw_l['so_id'] == oid][['line_number', 'product_id', 'quantity', 'unit_price', 'line_total']].to_string())
print('Sum of raw line_total:', raw_l.loc[raw_l['so_id'] == oid, 'line_total'].sum())
print('Same order in saved sales table: lines', (sales['so_id'] == oid).sum(), '| sum of line_total', sales.loc[sales['so_id'] == oid, 'line_total'].sum())

Rebuilding from the raw CSV files (fresh read, not the notebook tables)
Raw rows: header 20000 | lines 130402 | products 30
Delivered orders: 18033 | Delivered lines: 117656 | units: 1234939 | revenue: 22958987430

Products rebuilt from raw files: 30
Products where revenue differs from the saved table: 0
Products where class differs from the saved table: 0

Lines in raw rebuild: 117656 | lines in saved sales table: 117656
Lines only in the raw rebuild: 0
Lines only in the saved table: 0
Lines compared field by field: 117656
Lines where product, quantity, price or line total differs: 0

P015 Engine Block EB-600
  raw files : lines 3851 | units 40815 | revenue 4856985000 | share 21.16% | class A
  saved     : units 40815 | revenue 4856985000 | share 21.16 | class A
P022 Control Valve CV-75
  raw files : lines 3924 | units 41297 | revenue 697919300 | share 3.04% | class A
  saved     : units 41297 | revenue 697919300 | share 3.04 | class A
P005 Fuel Injector FI-220
  raw files : lines 389

In [56]:
spots = ['P015','P014','P022','P009','P005','P025','P026']

# rebuild the ranking from the raw files, Delivered orders only
deliv = raw_h.loc[raw_h['order_status'] == 'Delivered', 'so_id']
d = raw_l[raw_l['so_id'].isin(deliv)]
total = d['line_total'].sum()
rk = d.groupby('product_id')['line_total'].sum().sort_values(ascending=False).reset_index()
rk['rank'] = rk.index + 1
rk['share'] = rk['line_total'] / total * 100
rk['cum_after'] = rk['share'].cumsum()
rk['cum_before'] = rk['cum_after'] - rk['share']
rk['cls'] = rk['cum_before'].apply(lambda x: 'A' if x < 80 else ('B' if x < 95 else 'C'))

# saved table: find its column names
saved = rev if 'product_id' in rev.columns else rev.reset_index()
rev_col = [c for c in saved.columns if 'revenue' in c.lower()][0]
cls_col = [c for c in saved.columns if 'class' in c.lower()][0]
cum_col = [c for c in saved.columns if 'cum' in c.lower() and 'before' not in c.lower()][0]
print("Saved columns used:", rev_col, "|", cls_col, "|", cum_col)
print("Raw Delivered lines:", len(d), "| raw revenue total:", total)

problems = 0
for p in spots:
    lines = d[d['product_id'] == p]
    units = lines['quantity'].sum()
    price = raw_p.loc[raw_p['product_id'] == p, 'unit_price'].iloc[0]
    by_hand = units * price
    raw_rev = lines['line_total'].sum()
    r = rk[rk['product_id'] == p].iloc[0]
    s = saved[saved['product_id'] == p].iloc[0]
    print("\n==", p, "| rank", int(r['rank']), "==")
    print(lines.head(3)[['so_id','line_number','quantity','unit_price','line_total']].to_string(index=False))
    print("units", units, "x list price", price, "=", by_hand, "| raw revenue", raw_rev, "| saved revenue", s[rev_col])
    print("share %", round(r['share'], 2), "| cumulative % (after)", round(r['cum_after'], 2), "| saved cumulative %", round(s[cum_col], 2))
    print("class from raw", r['cls'], "| saved class", s[cls_col])
    same = (by_hand == raw_rev == s[rev_col]) and abs(r['cum_after'] - s[cum_col]) < 0.01 and r['cls'] == s[cls_col]
    print("Result:", "same everywhere" if same else "DIFFERENT")
    problems += 0 if same else 1

print("\nProducts checked:", len(spots), "| products with a difference:", problems)

Saved columns used: revenue | abc_class | cum_share
Raw Delivered lines: 117656 | raw revenue total: 22958987430

== P015 | rank 1 ==
    so_id  line_number  quantity  unit_price  line_total
SO-431510            1        13      119000     1547000
SO-495859            4        20      119000     2380000
SO-508292            5        14      119000     1666000
units 40815 x list price 119000 = 4856985000 | raw revenue 4856985000 | saved revenue 4856985000
share % 21.16 | cumulative % (after) 21.16 | saved cumulative % 21.16
class from raw A | saved class A
Result: same everywhere

== P014 | rank 2 ==
    so_id  line_number  quantity  unit_price  line_total
SO-494369           12        16       83500     1336000
SO-835154            9        16       83500     1336000
SO-532392            7         1       83500       83500
units 40574 x list price 83500 = 3387929000 | raw revenue 3387929000 | saved revenue 3387929000
share % 14.76 | cumulative % (after) 35.91 | saved cumulative % 35.91

In [57]:
g = sales.groupby('product_id')['line_grand_total'].sum().sort_values(ascending=False).reset_index()
tot = g['line_grand_total'].sum()
g['share'] = g['line_grand_total'] / tot * 100
g['cum_before'] = g['share'].cumsum() - g['share']
g['class_gst'] = g['cum_before'].apply(lambda x: 'A' if x < 80 else ('B' if x < 95 else 'C'))

saved = rev if 'product_id' in rev.columns else rev.reset_index()
cls_col = [c for c in saved.columns if 'class' in c.lower()][0]
m = g.merge(saved[['product_id', cls_col]], on='product_id')

print("Total revenue with GST:", round(tot, 2))
print("Products compared:", len(m))
print("Classes with GST -> A:", (m['class_gst'] == 'A').sum(), "| B:", (m['class_gst'] == 'B').sum(), "| C:", (m['class_gst'] == 'C').sum())
changed = m[m['class_gst'] != m[cls_col]]
print("Products whose class changes:", len(changed))
if len(changed) > 0:
    print(changed[['product_id', cls_col, 'class_gst']].to_string(index=False))

Total revenue with GST: 29253993162.4
Products compared: 30
Classes with GST -> A: 10 | B: 7 | C: 13
Products whose class changes: 0


In [58]:
u = sales.groupby('product_id')['quantity'].sum().sort_values(ascending=False).reset_index()
tu = u['quantity'].sum()
u['share'] = u['quantity'] / tu * 100
u['cum_before'] = u['share'].cumsum() - u['share']
u['class_units'] = u['cum_before'].apply(lambda x: 'A' if x < 80 else ('B' if x < 95 else 'C'))

saved = rev if 'product_id' in rev.columns else rev.reset_index()
cls_col = [c for c in saved.columns if 'class' in c.lower()][0]
m = u.merge(saved[['product_id', cls_col]], on='product_id')

print("Total units:", tu)
print("Smallest share % per product:", round(u['share'].min(), 2), "| largest:", round(u['share'].max(), 2))
print("Products needed to reach 80% of units:", (u['cum_before'] < 80).sum())
print("Products needed to reach 95% of units:", (u['cum_before'] < 95).sum())
print("Classes by units -> A:", (m['class_units'] == 'A').sum(), "| B:", (m['class_units'] == 'B').sum(), "| C:", (m['class_units'] == 'C').sum())
print("Products whose class differs from our revenue class:", (m['class_units'] != m[cls_col]).sum())
print("Our A products (by revenue) that are also A by units:", ((m[cls_col] == 'A') & (m['class_units'] == 'A')).sum(), "of", (m[cls_col] == 'A').sum())

Total units: 1234939
Smallest share % per product: 3.2 | largest: 3.46
Products needed to reach 80% of units: 24
Products needed to reach 95% of units: 29
Classes by units -> A: 24 | B: 5 | C: 1
Products whose class differs from our revenue class: 17
Our A products (by revenue) that are also A by units: 10 of 10


In [59]:
t5 = sales.groupby('product_id').agg(units=('quantity', 'sum'), revenue=('line_total', 'sum')).reset_index()
t5 = t5.merge(products[['product_id', 'unit_price']], on='product_id')

print("Products:", len(t5))
print("Rank correlation, list price vs revenue:", round(t5['unit_price'].corr(t5['revenue'], method='spearman'), 3))
print("Rank correlation, units vs revenue:", round(t5['units'].corr(t5['revenue'], method='spearman'), 3))
print("Units: smallest", t5['units'].min(), "| largest", t5['units'].max(), "| largest / smallest:", round(t5['units'].max() / t5['units'].min(), 2))
print("List price: smallest", t5['unit_price'].min(), "| largest", t5['unit_price'].max(), "| largest / smallest:", round(t5['unit_price'].max() / t5['unit_price'].min(), 1))
print("Revenue: smallest", t5['revenue'].min(), "| largest", t5['revenue'].max(), "| largest / smallest:", round(t5['revenue'].max() / t5['revenue'].min(), 1))

top10_price = set(t5.nlargest(10, 'unit_price')['product_id'])
a_set = set(saved.loc[saved[cls_col] == 'A', 'product_id'])
print("Our A products that are among the 10 highest list prices:", len(top10_price & a_set), "of", len(a_set))

Products: 30
Rank correlation, list price vs revenue: 0.999
Rank correlation, units vs revenue: 0.171
Units: smallest 39486 | largest 42702 | largest / smallest: 1.08
List price: smallest 310 | largest 119000 | largest / smallest: 383.9
Revenue: smallest 12526170 | largest 4856985000 | largest / smallest: 387.7
Our A products that are among the 10 highest list prices: 10 of 10


In [60]:
print("Table used: sales (Delivered lines) | rows:", len(sales))
cols = ['so_id', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount']
print("\nEmpty values per column:")
print(sales[cols].isnull().sum().to_string())
print("\nMin, max, zeros and negatives:")
for c in ['quantity', 'unit_price', 'line_total']:
    print(c, "| min", sales[c].min(), "| max", sales[c].max(), "| zeros", (sales[c] == 0).sum(), "| negatives", (sales[c] < 0).sum())
print("\nso_h empty values (so_id, order_status):", so_h[['so_id', 'order_status']].isnull().sum().sum())
print("products empty values (product_id, product_name, unit_price, gst_rate):", products[['product_id', 'product_name', 'unit_price', 'gst_rate']].isnull().sum().sum())

Table used: sales (Delivered lines) | rows: 117656

Empty values per column:
so_id         0
product_id    0
quantity      0
unit_price    0
gst_rate      0
line_total    0
gst_amount    0

Min, max, zeros and negatives:
quantity | min 1 | max 20 | zeros 0 | negatives 0
unit_price | min 310 | max 119000 | zeros 0 | negatives 0
line_total | min 310 | max 2380000 | zeros 0 | negatives 0

so_h empty values (so_id, order_status): 0
products empty values (product_id, product_name, unit_price, gst_rate): 0
